# 第四模块：因子评价与筛选

在 `labels&process.ipynb` 成功运行后执行本 notebook。
固定模型数据集版本及其关联的 Alpha158 原始因子版本，复用第三模块的标签、股票范围、
缺失阈值和时间切分。对**未填补、未截尾的原始因子**评价，保持自然缺失和并列值。

输出逐日 Pearson IC / Spearman RankIC、非年化 ICIR、覆盖率、常数截面比例、
月度稳定性、五分组收益和训练期冗余相关性，形成可复用的筛选名单。
筛选门槛、排名、因子方向和去冗余仅依据 train；valid 只做诊断，默认不评价 test。
不自动训练模型，不改写前三个 notebook、原始数据或模型数据分区。

横截面相关评价可参考 [Qlib 因子评价实现](https://github.com/microsoft/qlib/blob/main/qlib/contrib/eva/alpha.py)。
本模块的分组差为 `Q5-Q1`，与部分框架按资金归一化的多空收益定义不同；它不是可交易策略的回测净收益。

## 1. 配置与评价规则

依赖 NumPy、pandas、SciPy、matplotlib；读取现有 Parquet 需 pyarrow。
`MODEL_RUN=None` 在启动时固定 model_dataset 的 latest；因子源跟随该模型运行的来源，
不独立追逐 alpha158/latest。默认分析整个训练、验证截面，不抽样计算 IC。

因子按列分批读取，避免同时保留全市场 158 列；`FACTOR_BATCH_SIZE` 越小内存越低、读盘次数越多。
去冗余相关性单独使用训练期最多 100,000 个稳定哈希样本：先在完整日截面算百分位排名，
再抽取训练行计算排名间的 Pearson 相关。这是抽样的训练排名相关，不是逐日等权相关的精确均值。

以下门槛是首轮研究配置，不保证存在通过的因子；不满足条件时不会自动降低门槛或用验证结果补选。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import math
import shutil
import uuid
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, false_discovery_control
from IPython.display import display

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'feature.ipynb').exists() and (p / 'data').is_dir()),
                   Path(r'E:\document-prefer\python for finanace\multi-model-bactest'))
MODEL_ROOT = PROJECT_DIR / 'data' / 'processed' / 'model_dataset'
ALPHA_ROOT = PROJECT_DIR / 'data' / 'processed' / 'alpha158'
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_ROOT = PROJECT_DIR / 'data' / 'processed' / 'factor_selection'
MODEL_RUN = None
FACTOR_BATCH_SIZE = 8
MIN_CROSS_SECTION = 30
N_GROUPS = 5
MIN_COVERAGE = 0.95
MIN_IC_DAYS = 120
MIN_ABS_RANK_IC = 0.01
MIN_ABS_RANK_ICIR = 0.05
MIN_MONTH_POSITIVE = 0.55
MAX_CONSTANT_DAY_FRACTION = 0.10
CORR_SAMPLE_MAX_ROWS = 100_000
MIN_CORR_PAIRS = 200
MAX_PAIR_CORR = 0.90
MAX_FACTORS = 40
FDR_METHOD = 'by'      # BY 允许更一般的检验依赖；'bh' 适用条件更强
MAX_FDR_Q = None       # 默认仅报告；例如 0.10 可增加近似显著性门槛
EVALUATE_TEST = False  # 若设 True，仅作固定训练筛选规则的额外诊断
RUN_SELECTION = True
SCHEMA_VERSION = 1
plt.rcParams['figure.dpi'] = 110
print(f'项目：{PROJECT_DIR}\n因子批次宽度：{FACTOR_BATCH_SIZE}；测试期评价：{EVALUATE_TEST}')

def analysis_config():
    result = dict(batch_size=FACTOR_BATCH_SIZE, min_cross_section=MIN_CROSS_SECTION,
                  n_groups=N_GROUPS, min_coverage=MIN_COVERAGE, min_ic_days=MIN_IC_DAYS,
                  min_abs_rank_ic=MIN_ABS_RANK_IC, min_abs_rank_icir=MIN_ABS_RANK_ICIR,
                  min_month_positive=MIN_MONTH_POSITIVE,
                  max_constant_day_fraction=MAX_CONSTANT_DAY_FRACTION,
                  corr_sample_max_rows=CORR_SAMPLE_MAX_ROWS, min_corr_pairs=MIN_CORR_PAIRS,
                  max_pair_corr=MAX_PAIR_CORR, max_factors=MAX_FACTORS,
                  fdr_method=FDR_METHOD, max_fdr_q=MAX_FDR_Q, evaluate_test=EVALUATE_TEST)
    if result['batch_size'] < 1 or result['min_cross_section'] < 3 or result['n_groups'] < 2:
        raise ValueError('批次宽度、截面样本数或分组数无效')
    if result['min_ic_days'] < 2 or result['min_corr_pairs'] < 3 or result['max_factors'] < 1:
        raise ValueError('有效日数、相关样本数或最大因子数无效')
    if result['corr_sample_max_rows'] < result['min_corr_pairs']:
        raise ValueError('相关抽样上限小于最小成对样本数')
    for key in ('min_coverage', 'min_month_positive', 'max_constant_day_fraction', 'max_pair_corr'):
        if not 0 <= result[key] <= 1:
            raise ValueError(f'{key} 必须位于 [0,1]')
    if result['min_abs_rank_ic'] < 0 or result['min_abs_rank_icir'] < 0:
        raise ValueError('IC/ICIR 门槛不能为负')
    if result['fdr_method'] not in ('bh', 'by'):
        raise ValueError('fdr_method 必须为 bh 或 by')
    if result['max_fdr_q'] is not None and not 0 < result['max_fdr_q'] <= 1:
        raise ValueError('max_fdr_q 需在 (0,1] 或为 None')
    return result

config = analysis_config()

## 2. 固定数据版本及来源校验

读取模型运行中的 `config.json` 和原始因子来源，验证元数据哈希、158 列顺序、标签周期和进入延迟。
原始因子输入分区按股票划分；同一股票跨多个分区会报错，避免重复截面样本。
处理中会复核输入大小、修改时间和元数据，防止两个读取阶段之间混入其他版本。

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_model(root, run=None):
    root = Path(root)
    if run is not None:
        run = Path(run)
        return run if run.is_absolute() else root / run
    pointer = root / 'latest_run.json'
    if not pointer.exists():
        raise FileNotFoundError('先运行 labels&process.ipynb 生成正式模型数据集')
    path = (root / read_json(pointer)['run_dir']).resolve()
    if path.parent != root.resolve():
        raise ValueError('模型运行指针越出根目录')
    return path

def inspect_inputs(model_run, alpha_root=ALPHA_ROOT):
    model_run = Path(model_run).resolve()
    model_meta = read_json(model_run / 'metadata.json')
    model_config = read_json(model_run / 'config.json')
    names = read_json(model_run / 'feature_names.json')
    source_run = Path(model_meta['source_run'])
    if not source_run.exists():
        source_run = Path(alpha_root) / source_run.name
    source_run = source_run.resolve()
    source_meta = read_json(source_run / 'metadata.json')
    if sha256(source_run / 'metadata.json') != model_meta['source_metadata_sha256']:
        raise ValueError('模型运行对应的原始因子元数据哈希不符')
    if names != read_json(source_run / 'feature_names.json') or len(names) != len(set(names)) or len(names) != 158:
        raise ValueError('原始因子与模型数据集的 158 列名称或顺序不一致')
    expected_label = 'LABEL0' if model_config['target_horizon'] == 1 else f'LABEL{model_config["target_horizon"]}'
    if expected_label != model_meta['target_name']:
        raise ValueError('模型元数据的目标标签不一致')
    if model_config['target_horizon'] not in source_meta['label_horizons'] or model_config['entry_lag'] != source_meta['label_entry_lag']:
        raise ValueError('所选目标与原始因子标签配置不一致')
    previous_end = None
    if list(model_config['splits']) != ['train', 'valid', 'test']:
        raise ValueError('模型时间切分顺序无效')
    for split, (start, end) in model_config['splits'].items():
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        if start > end or (previous_end is not None and start <= previous_end):
            raise ValueError('模型时间切分存在重叠或倒序')
        previous_end = end
    pattern = 'part-*.parquet' if source_meta['format'] == 'parquet' else 'part-*.csv.gz'
    if source_meta['format'] not in ('parquet', 'csv.gz'):
        raise ValueError('未知原始因子存储格式')
    if source_meta['format'] == 'parquet' and importlib.util.find_spec('pyarrow') is None:
        raise ImportError('读取现有 Parquet 需要 pyarrow，请在当前内核运行 %pip install pyarrow')
    paths = sorted(source_run.glob(pattern))
    if not paths or len(paths) != source_meta['partitions']:
        raise ValueError('原始因子分区数量不符')
    manifest = [{'file': path.name, 'bytes': path.stat().st_size, 'mtime_ns': path.stat().st_mtime_ns} for path in paths]
    return dict(model_run=model_run, source_run=source_run, model_meta=model_meta,
                model_config=model_config, source_meta=source_meta, names=names, paths=paths,
                manifest=manifest, model_metadata_sha256=sha256(model_run / 'metadata.json'),
                model_config_sha256=sha256(model_run / 'config.json'))

def assert_inputs_unchanged(inputs):
    if sha256(inputs['model_run'] / 'metadata.json') != inputs['model_metadata_sha256']:
        raise RuntimeError('模型运行元数据发生变化')
    if sha256(inputs['model_run'] / 'config.json') != inputs['model_config_sha256']:
        raise RuntimeError('模型运行配置发生变化')
    if sha256(inputs['source_run'] / 'metadata.json') != inputs['model_meta']['source_metadata_sha256']:
        raise RuntimeError('因子运行元数据发生变化')
    for item in inputs['manifest']:
        stat = (inputs['source_run'] / item['file']).stat()
        if stat.st_size != item['bytes'] or stat.st_mtime_ns != item['mtime_ns']:
            raise RuntimeError(f'因子分区发生变化：{item["file"]}')

def read_raw(path, columns, inputs):
    if inputs['source_meta']['format'] == 'parquet':
        frame = pd.read_parquet(path, columns=columns)
    else:
        frame = pd.read_csv(path, usecols=['datetime', 'instrument'] + columns, dtype={'instrument': str})
        frame['datetime'] = pd.to_datetime(frame['datetime'], errors='raise')
        frame = frame.set_index(['datetime', 'instrument'])
    if frame.index.names != ['datetime', 'instrument'] or frame.index.has_duplicates:
        raise ValueError(f'原始因子索引无效：{path}')
    dates = pd.to_datetime(frame.index.get_level_values('datetime'), errors='raise')
    codes = frame.index.get_level_values('instrument')
    if dates.isna().any() or pd.isna(codes).any():
        raise ValueError('原始因子日期或代码缺失')
    frame.index = pd.MultiIndex.from_arrays([dates, codes.astype(str)], names=['datetime', 'instrument'])
    return frame

inputs = None
try:
    inputs = inspect_inputs(resolve_model(MODEL_ROOT, MODEL_RUN))
    print(f'模型版本：{inputs["model_run"].name}\n因子版本：{inputs["source_run"].name}')
    print(f'评价标签：{inputs["model_meta"]["target_name"]}；原始收益用于 IC 和分组评价。')
except FileNotFoundError as exc:
    print(f'上游尚未就绪：{exc}；可以继续执行独立自检。')

## 3. 构建与模型一致的评价样本

先逐原始分区检查所有 158 列的缺失比例，然后按第三模块的股票范围、时间区间、
标签存在性及标签结束日期重新确定监督样本。与第三模块报告的监督样本数交叉核对。
不把原始辅助列当作因子，不使用填补后的数据估计因子覆盖率。

In [ ]:
def load_calendar(inputs, raw_dir=RAW_DIR):
    values = []
    paths = sorted((Path(raw_dir) / 'basic').glob('trade_dates_*.csv'))
    if not paths:
        raise FileNotFoundError('缺少交易日历')
    for path in paths:
        values.extend(pd.read_csv(path, dtype=str, usecols=['trade_date']).trade_date.tolist())
    cal = pd.DatetimeIndex(pd.to_datetime(values, format='%Y%m%d')).unique().sort_values()
    start, end = inputs['source_meta']['start_date'], inputs['source_meta']['end_date']
    return cal[(cal >= pd.Timestamp(start)) & (cal <= pd.Timestamp(end))]

def build_evaluation_base(inputs, config, calendar):
    label = inputs['model_meta']['target_name']
    names, upstream = inputs['names'], inputs['model_config']
    active = ['train', 'valid'] + (['test'] if config['evaluate_test'] else [])
    pieces, partitions, audit, seen = [], {}, [], set()
    date_series = pd.Series(calendar, index=calendar)
    for part_id, path in enumerate(inputs['paths']):
        frame = read_raw(path, names + [label, 'label_entry_date', f'{label}_end_date'], inputs)
        codes = set(frame.index.get_level_values('instrument'))
        if codes & seen:
            raise ValueError('同一股票跨原始分区重复，不符合上游按股票分批的契约')
        seen.update(codes)
        if upstream['instruments'] is not None:
            frame = frame.loc[frame.index.get_level_values('instrument').isin(upstream['instruments'])]
        dates = pd.DatetimeIndex(frame.index.get_level_values('datetime'))
        split_values = np.full(len(frame), 'outside', dtype=object)
        for split in active:
            start, end = map(pd.Timestamp, upstream['splits'][split])
            split_values[(dates >= start) & (dates <= end)] = split
        frame = frame.loc[split_values != 'outside'].copy()
        split_values = split_values[split_values != 'outside']
        dates = pd.DatetimeIndex(frame.index.get_level_values('datetime'))
        if not dates.isin(calendar).all():
            raise ValueError('因子日期不在交易日历中')
        for field, lag in [('label_entry_date', upstream['entry_lag']),
                           (f'{label}_end_date', upstream['entry_lag'] + upstream['target_horizon'])]:
            actual = pd.to_datetime(frame[field])
            expected = pd.Series(date_series.shift(-lag).reindex(dates).to_numpy(), index=frame.index)
            if not (actual.eq(expected) | (actual.isna() & expected.isna())).all():
                raise ValueError(f'{field} 与交易日历不符')
        X = frame[names].replace([np.inf, -np.inf], np.nan)
        y = pd.to_numeric(frame[label], errors='raise')
        if np.isinf(y.to_numpy()).any():
            raise ValueError('原始收益标签存在无穷值')
        ends = pd.to_datetime(frame[f'{label}_end_date'])
        feature_ok = X.isna().mean(axis=1).le(upstream['max_row_missing'])
        boundary = pd.to_datetime(pd.Series(split_values, index=frame.index).map(
            {split: upstream['splits'][split][1] for split in active}))
        label_ok = y.notna() & ends.le(boundary)
        eligible = feature_ok & label_ok
        local = pd.DataFrame({'y': y.loc[eligible].astype(float),
                              'split': split_values[eligible.to_numpy()], 'part_id': part_id},
                             index=frame.index[eligible])
        partitions[part_id] = local.index
        pieces.append(local)
        for split in active:
            mask = split_values == split
            audit.append({'split': split, 'rows': int(mask.sum()),
                          'eligible': int((eligible.to_numpy() & mask).sum())})
    base = pd.concat(pieces)
    if base.empty or base.index.has_duplicates:
        raise ValueError('评价样本为空或日期股票主键重复')
    summary = pd.DataFrame(audit).groupby('split').sum()
    expected = pd.read_csv(inputs['model_run'] / 'split_summary.csv').set_index('split')
    for split in active:
        if split not in summary.index or summary.loc[split, 'eligible'] == 0:
            raise ValueError(f'{split} 无可用评价样本')
        if int(summary.loc[split, 'eligible']) != int(expected.loc[split, 'supervised_eligible']):
            raise ValueError(f'{split} 监督样本数与第三模块报告不一致')
    display(summary)
    return base, partitions, summary

def read_factor_batch(inputs, partitions, names):
    pieces = []
    for part_id, path in enumerate(inputs['paths']):
        wanted = partitions[part_id]
        if wanted.empty:
            continue
        frame = read_raw(path, names, inputs)
        if not wanted.isin(frame.index).all():
            raise ValueError('分批读取时样本索引丢失')
        pieces.append(frame.reindex(wanted)[names].replace([np.inf, -np.inf], np.nan).astype('float32'))
    return pd.concat(pieces)

## 4. 逐日评价和统计定义

每个因子只在自身与收益均有效的股票对上计算 IC 和 RankIC；并列值用平均排名。
有效股票对不足门槛、因子常数或收益常数时，相应相关系数保留 NaN，不填 0。
每天的 IC 等权汇总，ICIR 定义为日 IC 均值/样本标准差，不年化。

按因子百分位平均排名分组，Q1 最低、Q5 最高；并列值可能使分组不等大或缺少端点组。
分组收益是该组有效股票未来收益的等权均值，端点组缺失时不强行构造 Q5-Q1。
多日标签存在收益重叠，分组差不复利、不当作净值，也不含费用、风险中性化或成交限制。

RankIC 均值另提供 Bartlett 核 HAC 标准误及正态近似 p 值，滞后阶数至少为 5 且不小于 h-1；
缺失日保留在交易日位置，协方差计算忽略其残差贡献。
p 值按全部因子做 BY/BH 调整（[SciPy 说明](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html)），
只作为探索性证据；HAC/正态近似、因子依赖和重复研究都限制其解释，默认不作为硬筛选条件。

In [ ]:
def pearson(x, y):
    dx, dy = x - x.mean(), y - y.mean()
    denominator = np.sqrt(np.dot(dx, dx) * np.dot(dy, dy))
    return float(np.clip(np.dot(dx, dy) / denominator, -1, 1)) if denominator > 1e-15 else np.nan

def cross_section_metrics(values, returns, config):
    valid = np.isfinite(values) & np.isfinite(returns)
    x, y = np.asarray(values[valid], float), np.asarray(returns[valid], float)
    n = len(x)
    out = dict(n_total=len(values), n_valid=n, constant=False, ic=np.nan, rank_ic=np.nan,
               spread=np.nan, market_return=float(y.mean()) if n else np.nan)
    out.update({f'q{q}': np.nan for q in range(1, config['n_groups']+1)})
    ranks = np.full(len(values), np.nan)
    if n:
        rx = rankdata(x, method='average')
        ranks[valid] = rx / n
    if n < config['min_cross_section']:
        return out, ranks
    if np.ptp(x) <= 1e-12:
        out['constant'] = True
        return out, ranks
    out['ic'] = pearson(x, y)
    out['rank_ic'] = pearson(rx, rankdata(y, method='average'))
    buckets = np.minimum(np.ceil(rx / n * config['n_groups']).astype(int), config['n_groups'])
    for q in range(1, config['n_groups']+1):
        selected = y[buckets == q]
        if len(selected):
            out[f'q{q}'] = float(selected.mean())
    out['spread'] = out[f'q{config["n_groups"]}'] - out['q1']
    return out, ranks

def hac_mean_stats(series, lags):
    values = np.asarray(series, float)
    valid = np.isfinite(values)
    n = int(valid.sum())
    if n < 3:
        return np.nan, np.nan, np.nan
    average = float(values[valid].mean())
    residual = np.where(valid, values-average, 0.)
    lags = min(int(lags), len(values)-1)
    long_run = float(np.dot(residual, residual) / n)
    for lag in range(1, lags+1):
        long_run += 2*(1-lag/(lags+1))*float(np.dot(residual[lag:], residual[:-lag])/n)
    se = math.sqrt(max(long_run, 0.) / n)
    if se <= 1e-15:
        return se, math.copysign(np.inf, average) if average else 0., 0. if average else 1.
    t = average / se
    return se, t, math.erfc(abs(t) / math.sqrt(2))

def evaluate_batches(inputs, base, partitions, config):
    names = inputs['names']
    train_index = base.index[base['split'].eq('train')]
    keys = pd.Series(pd.util.hash_pandas_object(train_index, index=True).to_numpy(), index=train_index)
    sample_index = keys.nsmallest(config['corr_sample_max_rows']).index
    rank_sample = pd.DataFrame(np.nan, index=sample_index, columns=names, dtype='float32')
    groups = base.groupby(level='datetime', sort=True).indices
    y = base.y.to_numpy(dtype=float)
    splits = base['split'].to_numpy()
    records = []
    for offset in range(0, len(names), config['batch_size']):
        batch_names = names[offset:offset+config['batch_size']]
        batch = read_factor_batch(inputs, partitions, batch_names)
        if not batch.index.equals(base.index):
            raise ValueError('因子批次与收益索引不一致')
        values = batch.to_numpy()
        columns = np.arange(offset, offset+len(batch_names))
        for day, positions in groups.items():
            indices = batch.index[positions]
            sample_positions = sample_index.get_indexer(indices)
            is_sample = sample_positions >= 0
            for local_id, factor in enumerate(batch_names):
                metrics, ranks = cross_section_metrics(values[positions, local_id], y[positions], config)
                metrics.update(datetime=pd.Timestamp(day), factor=factor, split=splits[positions[0]])
                records.append(metrics)
                if is_sample.any():
                    rank_sample.iloc[sample_positions[is_sample], columns[local_id]] = ranks[is_sample].astype('float32')
        print(f'完成因子 {min(offset+config["batch_size"], len(names))}/{len(names)}', flush=True)
        assert_inputs_unchanged(inputs)
    daily = pd.DataFrame(records).sort_values(['split', 'factor', 'datetime'])
    return daily, rank_sample

def summarize_metrics(daily, names, calendar, inputs, config):
    rows, monthly_rows = [], []
    horizon = inputs['model_config']['target_horizon']
    for (split, factor), block in daily.groupby(['split', 'factor'], sort=False):
        start, end = map(pd.Timestamp, inputs['model_config']['splits'][split])
        days = calendar[(calendar >= start) & (calendar <= end)]
        block = block.set_index('datetime').reindex(days)
        ic, rank_ic = block.ic.dropna(), block.rank_ic.dropna()
        mean_ic = float(ic.mean())
        mean_rank = float(rank_ic.mean())
        std_ic, std_rank = float(ic.std(ddof=1)), float(rank_ic.std(ddof=1))
        def ratio(mean, std):
            if np.isnan(std) or np.isnan(mean):
                return np.nan
            return mean/std if std > 1e-15 else (math.copysign(np.inf, mean) if mean else 0.)
        se, t_value, p = hac_mean_stats(block.rank_ic.to_numpy(), max(5, horizon-1))
        month = block.rank_ic.groupby(block.index.to_period('M')).mean()
        for period, value in month.items():
            monthly_rows.append(dict(split=split, factor=factor, month=str(period), rank_ic=value))
        total = block.n_total.sum()
        row = dict(split=split, factor=factor, days=len(block), ic_days=len(ic), rank_ic_days=len(rank_ic),
                   coverage=block.n_valid.sum()/total if total else np.nan,
                   constant_day_fraction=(block.constant.fillna(False).astype(bool).sum()/block.n_valid.ge(config['min_cross_section']).sum()
                                          if block.n_valid.ge(config['min_cross_section']).any() else np.nan),
                   ic_mean=mean_ic, ic_std=std_ic, ic_ir=ratio(mean_ic, std_ic),
                   rank_ic_mean=mean_rank, rank_ic_std=std_rank, rank_ic_ir=ratio(mean_rank, std_rank),
                   rank_ic_hac_se=se, rank_ic_hac_t=t_value, rank_ic_p=p,
                   spread_mean=block.spread.mean(), spread_days=block.spread.notna().sum())
        row.update({f'q{q}_mean': block[f'q{q}'].mean() for q in range(1, config['n_groups']+1)})
        rows.append(row)
    report = pd.DataFrame(rows)
    for split in report['split'].unique():
        mask = report['split'].eq(split)
        p_values = report.loc[mask].set_index('factor').rank_ic_p.reindex(names).fillna(1.)
        adjusted = pd.Series(false_discovery_control(p_values.to_numpy(), method=config['fdr_method']), index=names)
        report.loc[mask, 'rank_ic_q'] = report.loc[mask, 'factor'].map(adjusted).to_numpy()
    monthly = pd.DataFrame(monthly_rows)
    train = report.loc[report['split'].eq('train')].set_index('factor')
    direction = pd.Series(np.where(train.rank_ic_mean.ge(0), 1, -1), index=train.index)
    report['direction'] = report.factor.map(direction)
    report['signed_rank_ic_mean'] = report.rank_ic_mean * report.direction
    report['signed_spread_mean'] = report.spread_mean * report.direction
    monthly['direction'] = monthly.factor.map(direction)
    monthly['signed_rank_ic'] = monthly.rank_ic * monthly.direction
    stability = monthly.dropna(subset=['signed_rank_ic']).groupby(['split','factor']).signed_rank_ic.agg(
        month_positive_fraction=lambda s: float(s.gt(0).mean()), months='size')
    report = report.merge(stability, on=['split','factor'], how='left')
    return report, monthly

## 5. 训练期筛选和去冗余

先检查训练覆盖率、有效 RankIC 日数、常数日比例、|RankIC 均值|、|RankICIR| 和月度方向一致率。
方向由训练 RankIC 均值的正负决定，验证期不会重新翻转方向。
通过门槛后按训练 |RankIC 均值|、|RankICIR| 排序，依次保留与已选因子绝对相关不超过阈值的因子。
成对抽样不足或相关性不可计算时不盲目认定“独立”。

输出每个因子的具体决策原因；最多保留指定数量，可能少于该数量甚至为空。
这是单因子筛选，可能剔除只有非线性交互才有用的变量，因此后续模型应同时比较全量 158 列和筛选子集。

In [ ]:
def training_correlations(rank_sample):
    correlation = rank_sample.corr(method='pearson', min_periods=3)
    valid = rank_sample.notna().to_numpy(dtype='float32')
    counts = pd.DataFrame(np.rint(valid.T @ valid).astype('int64'),
                          index=rank_sample.columns, columns=rank_sample.columns)
    return correlation, counts

def select_training_factors(report, correlation, counts, names, config):
    train = report.loc[report['split'].eq('train')].set_index('factor').reindex(names)
    if train.index.has_duplicates or train.rank_ic_mean.isna().all():
        raise ValueError('没有可评价的训练因子')
    decisions = train.copy()
    decisions['candidate'] = False
    decisions['selected'] = False
    decisions['reason'] = ''
    decisions['redundant_with'] = ''
    decisions['redundancy_corr'] = np.nan
    for factor, row in train.iterrows():
        failures = []
        rules = [
            ('low_coverage', pd.notna(row.coverage) and row.coverage >= config['min_coverage']),
            ('few_ic_days', row.rank_ic_days >= config['min_ic_days']),
            ('too_many_constant_days', pd.notna(row.constant_day_fraction) and row.constant_day_fraction <= config['max_constant_day_fraction']),
            ('weak_rank_ic', pd.notna(row.rank_ic_mean) and abs(row.rank_ic_mean) >= config['min_abs_rank_ic']),
            ('weak_rank_icir', pd.notna(row.rank_ic_ir) and abs(row.rank_ic_ir) >= config['min_abs_rank_icir']),
            ('unstable_months', pd.notna(row.month_positive_fraction) and row.month_positive_fraction >= config['min_month_positive']),
        ]
        if config['max_fdr_q'] is not None:
            rules.append(('fdr_not_passed', pd.notna(row.rank_ic_q) and row.rank_ic_q <= config['max_fdr_q']))
        failures = [reason for reason, passed in rules if not passed]
        decisions.loc[factor, 'candidate'] = not failures
        decisions.loc[factor, 'reason'] = ';'.join(failures)
    ordered = train.loc[decisions.candidate].assign(
        abs_rank_ic=lambda d: d.rank_ic_mean.abs(), abs_rank_icir=lambda d: d.rank_ic_ir.abs())
    ordered = ordered.reset_index().sort_values(['abs_rank_ic','abs_rank_icir','factor'], ascending=[False,False,True])
    selected = []
    for factor in ordered.factor:
        if len(selected) >= config['max_factors']:
            decisions.loc[factor, 'reason'] = 'max_factors_reached'
            continue
        if selected:
            paired = counts.loc[factor, selected]
            correlations = correlation.loc[factor, selected]
            if (paired < config['min_corr_pairs']).any() or correlations.isna().any():
                decisions.loc[factor, 'reason'] = 'insufficient_redundancy_evidence'
                continue
            strongest = correlations.abs().idxmax()
            if abs(correlations[strongest]) > config['max_pair_corr']:
                decisions.loc[factor, ['reason','redundant_with','redundancy_corr']] = [
                    'redundant', strongest, float(correlations[strongest])]
                continue
        selected.append(factor)
        decisions.loc[factor, 'selected'] = True
        decisions.loc[factor, 'reason'] = 'selected'
    decisions.index.name = 'factor'
    return selected, decisions.reset_index()

## 6. 报告、图表及模型接入接口

保存到 `data/processed/factor_selection/run_*`：

- `daily_metrics.parquet` / `.csv.gz`、`factor_report.csv`、`monthly_rank_ic.csv`；
- `train_correlation.csv`、`train_correlation_pair_counts.csv`、`sample_summary.csv`；
- `selection_decisions.csv`、`selected_factors.json`、`factor_directions.json`；
- `config.json`、`source.json`、`metadata.json` 和 `charts/*.png`。

只保存名单和评价产物，不复制或改写第三模块的数据。
名单保留训练评价排序；因子方向供解释和单因子分组使用，不自动改变多模型输入的符号。
读取器可核对模型运行版本，避免将针对其他标签或数据版本的名单误用。

In [ ]:
def write_json(path, content):
    Path(path).write_text(json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def save_charts(folder, daily, report, correlation, selected, config):
    folder.mkdir(parents=True, exist_ok=True)
    train = report.loc[report['split'].eq('train')].set_index('factor')
    top = selected[:6] or train.rank_ic_mean.abs().nlargest(6).index.tolist()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    for ax, split in zip(axes, ['train','valid']):
        for factor in top:
            block = daily.loc[daily['split'].eq(split) & daily.factor.eq(factor)].sort_values('datetime')
            direction = int(train.loc[factor, 'direction'])
            ax.plot(block.datetime, (block.rank_ic.fillna(0)*direction).cumsum(), label=factor)
        ax.set_title(f'{split}: cumulative signed RankIC')
        ax.set_ylabel('Sum of RankIC (not portfolio NAV)')
        ax.tick_params(axis='x', rotation=30)
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=7, ncol=2)
    fig.savefig(folder/'rank_ic_cumulative.png', dpi=200)
    display(fig)
    plt.close(fig)
    if top:
        factor = top[0]
        fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
        direction = int(train.loc[factor, 'direction'])
        columns = [f'q{q}_mean' for q in range(1, config['n_groups']+1)]
        if direction < 0:
            columns = columns[::-1]
        for split in ['train','valid']:
            row = report.loc[report['split'].eq(split) & report.factor.eq(factor)]
            if not row.empty:
                ax.plot(range(1,config['n_groups']+1), row.iloc[0][columns].to_numpy(dtype=float), marker='o', label=split)
        ax.set(title=f'{factor}: groups ordered by training direction',
               xlabel='Weak to strong training-direction score', ylabel='Mean forward return (before costs)')
        ax.legend()
        ax.grid(alpha=.2)
        fig.savefig(folder/'group_return_profile.png', dpi=200)
        display(fig)
        plt.close(fig)
    heatmap_names = selected[:25] or top
    if heatmap_names:
        fig, ax = plt.subplots(figsize=(max(6, len(heatmap_names)*.35), max(5,len(heatmap_names)*.35)), constrained_layout=True)
        im = ax.imshow(correlation.loc[heatmap_names, heatmap_names], vmin=-1, vmax=1, cmap='coolwarm')
        ax.set_xticks(range(len(heatmap_names)), heatmap_names, rotation=90, fontsize=7)
        ax.set_yticks(range(len(heatmap_names)), heatmap_names, fontsize=7)
        ax.set_title('Training cross-sectional rank correlation (sample)')
        fig.colorbar(im, ax=ax, shrink=.8)
        fig.savefig(folder/'train_correlation.png', dpi=200)
        display(fig)
        plt.close(fig)

def run_factor_selection(inputs, output_root=OUTPUT_ROOT, raw_dir=RAW_DIR, config=None):
    config = analysis_config() if config is None else dict(config)
    assert_inputs_unchanged(inputs)
    calendar = load_calendar(inputs, raw_dir)
    base, partitions, sample_summary = build_evaluation_base(inputs, config, calendar)
    daily, rank_sample = evaluate_batches(inputs, base, partitions, config)
    report, monthly = summarize_metrics(daily, inputs['names'], calendar, inputs, config)
    correlation, counts = training_correlations(rank_sample)
    selected, decisions = select_training_factors(report, correlation, counts, inputs['names'], config)
    assert_inputs_unchanged(inputs)
    root = Path(output_root).resolve()
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_id = f'run_{stamp}_{uuid.uuid4().hex[:8]}'
    staging, target = root/f'.{run_id}.tmp', root/run_id
    staging.mkdir()
    try:
        report.to_csv(staging/'factor_report.csv', index=False)
        monthly.to_csv(staging/'monthly_rank_ic.csv', index=False)
        correlation.to_csv(staging/'train_correlation.csv')
        counts.to_csv(staging/'train_correlation_pair_counts.csv')
        decisions.to_csv(staging/'selection_decisions.csv', index=False)
        sample_summary.to_csv(staging/'sample_summary.csv')
        output_format = 'parquet' if importlib.util.find_spec('pyarrow') else 'csv.gz'
        if output_format == 'parquet':
            daily.to_parquet(staging/'daily_metrics.parquet', index=False)
        else:
            daily.to_csv(staging/'daily_metrics.csv.gz', index=False, compression='gzip')
        train = report.loc[report['split'].eq('train')].set_index('factor')
        directions = {name: int(train.loc[name,'direction']) for name in selected}
        write_json(staging/'selected_factors.json', selected)
        write_json(staging/'factor_directions.json', directions)
        write_json(staging/'config.json', config)
        write_json(staging/'source.json', {
            'model_run': str(inputs['model_run']), 'model_metadata_sha256': inputs['model_metadata_sha256'],
            'model_config_sha256': inputs['model_config_sha256'], 'model_config': inputs['model_config'],
            'alpha_run': str(inputs['source_run']),
            'alpha_metadata_sha256': inputs['model_meta']['source_metadata_sha256'],
            'partitions': inputs['manifest'], 'features': inputs['names']})
        metadata = dict(schema_version=SCHEMA_VERSION, created_utc=stamp,
            selection_split='train', validation_used_for_selection=False, test_used_for_selection=False,
            evaluated_splits=sorted(base['split'].unique().tolist()), target_name=inputs['model_meta']['target_name'],
            target_horizon=inputs['model_config']['target_horizon'],
            model_run=str(inputs['model_run']), selected_count=len(selected), source_features=len(inputs['names']),
            corr_sample_rows=len(rank_sample), correlation_method='sampled_train_daily_percentile_rank_pearson',
            hac_lags=max(5,inputs['model_config']['target_horizon']-1), icir_annualized=False,
            p_value_method='Bartlett HAC mean SE and asymptotic normal approximation',
            pandas=pd.__version__, numpy=np.__version__, output_format=output_format)
        write_json(staging/'metadata.json', metadata)
        save_charts(staging/'charts', daily, report, correlation, selected, config)
        assert_inputs_unchanged(inputs)
        staging.rename(target)
        pointer = root/f'.latest-{uuid.uuid4().hex}.tmp'
        write_json(pointer, {'run_dir':target.name})
        pointer.replace(root/'latest_run.json')
    except Exception:
        if staging.is_dir() and staging.resolve().parent == root:
            shutil.rmtree(staging)
        raise
    if not selected:
        warnings.warn('没有因子通过当前训练筛选条件；已保存完整报告，不自动放宽阈值。')
    print(f'选中 {len(selected)}/{len(inputs["names"])} 个因子；输出：{target}')
    display(decisions.loc[decisions.selected, ['factor','direction','rank_ic_mean','rank_ic_ir','month_positive_fraction','reason']])
    validation = report.loc[report['split'].eq('valid') & report.factor.isin(selected)]
    display(validation[['factor','rank_ic_mean','signed_rank_ic_mean','signed_spread_mean','month_positive_fraction']])
    return dict(run_dir=target, selected=selected, decisions=decisions, report=report, correlation=correlation)

def load_selected_factors(run=None, model_run=None, root=OUTPUT_ROOT):
    root = Path(root)
    if run is None:
        run = root/read_json(root/'latest_run.json')['run_dir']
        if run.resolve().parent != root.resolve():
            raise ValueError('筛选运行指针越出根目录')
    run = Path(run)
    names = read_json(run/'selected_factors.json')
    source = read_json(run/'source.json')
    if not names:
        raise ValueError('筛选名单为空，请查阅 selection_decisions.csv')
    if len(names) != len(set(names)) or not set(names).issubset(source['features']):
        raise ValueError('筛选名单存在重复或未知因子')
    if model_run is not None:
        model_run = Path(model_run)
        if sha256(model_run/'metadata.json') != source['model_metadata_sha256'] or sha256(model_run/'config.json') != source['model_config_sha256']:
            raise ValueError('筛选名单与当前模型数据集版本不匹配')
    return names

def apply_factor_selection(X, run=None, model_run=None):
    selected = load_selected_factors(run, model_run)
    if not set(selected).issubset(X.columns):
        raise ValueError('模型 X 缺少已选因子')
    return X.loc[:, selected].copy()

## 7. 独立方法自检

验证相关系数、并列排名、常数和缺失截面、分组方向、HAC 均值误差，
并验证重复/反向因子只保留一个，以及任意改变验证期报告不会改变筛选名单。
合成数据仅用于自检，不写作正式研究结果。

In [ ]:
def run_self_checks():
    cfg = analysis_config()
    cfg.update(min_cross_section=10, min_ic_days=10, min_corr_pairs=20,
               min_coverage=.8, min_month_positive=.5, max_factors=3)
    x = np.arange(60,dtype=float)
    y = x/1000
    positive, ranks = cross_section_metrics(x,y,cfg)
    negative, _ = cross_section_metrics(-x,y,cfg)
    assert np.isclose(positive['ic'],1) and np.isclose(positive['rank_ic'],1)
    assert np.isclose(negative['rank_ic'],-1)
    assert positive['spread'] > 0 and negative['spread'] < 0
    flat, _ = cross_section_metrics(np.ones(60),y,cfg)
    assert flat['constant'] and np.isnan(flat['rank_ic'])
    missing, _ = cross_section_metrics(np.full(60,np.nan),y,cfg)
    assert missing['n_valid']==0 and np.isnan(missing['ic'])
    tied, _ = cross_section_metrics(np.repeat([1.,2.,3.],20), y, cfg)
    from scipy.stats import spearmanr
    assert np.isclose(tied['rank_ic'], spearmanr(np.repeat([1.,2.,3.],20),y).statistic)
    se, _, _ = hac_mean_stats(np.array([1.,2.,3.,4.]),0)
    assert np.isclose(se,np.std([1.,2.,3.,4.],ddof=0)/2)
    names = ['positive','duplicate','negative','constant']
    dates = pd.bdate_range('2022-01-03',periods=50)
    rng = np.random.default_rng(158)
    rows=[]
    for day in dates:
        for split in ['train','valid']:
            yy = x/1000+rng.normal(0,.02,60)
            for name,xx in [('positive',x),('duplicate',x),('negative',-x),('constant',np.ones(60))]:
                metrics,_=cross_section_metrics(xx,yy,cfg)
                metrics.update(datetime=day,factor=name,split=split)
                rows.append(metrics)
    daily=pd.DataFrame(rows)
    fake_inputs={'model_config':{'target_horizon':1,'splits':{
        'train':['2022-01-03','2022-03-31'],'valid':['2022-01-03','2022-03-31']}}}
    report,monthly=summarize_metrics(daily,names,pd.bdate_range('2022-01-03','2022-03-31'),fake_inputs,cfg)
    assert report.loc[report.factor.eq('constant'),'constant_day_fraction'].eq(1.).all()
    sample=pd.DataFrame({'positive':np.tile(x,4),'duplicate':np.tile(x,4),
                         'negative':-np.tile(x,4),'constant':1.})
    corr,counts=training_correlations(sample)
    selected,decisions=select_training_factors(report,corr,counts,names,cfg)
    assert len(selected)==1 and selected[0]!='constant'
    altered=report.copy()
    altered.loc[altered['split'].eq('valid'),'rank_ic_mean']=-1e5
    altered.loc[altered['split'].eq('valid'),'rank_ic_ir']=1e9
    assert selected==select_training_factors(altered,corr,counts,names,cfg)[0]
    assert (decisions.reason.eq('redundant')).sum()==2
    print('通过：IC、并列排名、分组、缺失/常数、HAC、冗余和验证期独立性自检。')
    return {'checks':'passed'}

self_check_result = run_self_checks()

## 8. 正式执行与结果使用

正式执行会先读取一次全部原始因子确定样本，再分列批次计算，耗时取决于机器和磁盘。
图表和评价报告先用于理解因子；valid 中若出现方向翻转，会保留在诊断中，不自动补选。
修改门槛属于新的研究实验，应保留旧运行并在独立样本上评价，避免反复看验证结果后挑选最优配置。

In [ ]:
selection_result = None
if RUN_SELECTION and inputs is not None:
    selection_result = run_factor_selection(inputs, config=config)
elif not RUN_SELECTION:
    print('RUN_SELECTION=False：已执行定义与自检，未生成正式筛选结果。')
else:
    print('正式筛选跳过：请先完成前三个 notebook。')

## 9. 后续模型使用示例

保持第三模块的 tree / linear 数据版本不变，只选择相同的列。模型之间同时记录“全量 Alpha158”
与“训练筛选子集”的对照，判断单因子筛选是否真正改善样本外预测和回测。

```python
# 在相应接口已定义的内核中调用；也可独立读取 selected_factors.json。
selection_run = selection_result['run_dir']
model_run = inputs['model_run']
# X_train, y_train = 第三模块的 load_model_data('train', view='tree', run=model_run)
# X_valid, y_valid = 第三模块的 load_model_data('valid', view='tree', run=model_run)
# X_train_selected = apply_factor_selection(X_train, selection_run, model_run)
# X_valid_selected = apply_factor_selection(X_valid, selection_run, model_run)
```

测试期默认为保留样本。后续使用已固定的名单训练/评估模型时，不应再依据测试效果调整因子。
这里只做量价单因子评价，尚未进行行业、市值中性化，也未模拟交易执行和费用。